# Phase ① (file_manager 분리) Toy 검증 노트북

**목적**: file_manager 분리 작업(① docs)의 인터페이스/데이터 흐름이 실제 인프라(MongoDB/Redis/llm-handler)에서 의도대로 동작하는지 단위 시연.

**가정**:
- file_manager 패키지는 아직 없음 → `service/app/models/file_meta.py`의 `FileMeta`를 stub으로 등록
- file_server에서 받은 파일은 `quickstarts/small_file.pdf`, `large_file.pdf`로 가정 (`fetch_file_bytes`는 mock해서 로컬 파일 read)
- llm_handler: 192.7.9.45:8100 (실제)
- mdb_handler: 192.7.9.45:8300 (실제), database = `kw_ai_dev`
- redis: 192.7.9.45:7617 (실제, Predixy 경유)

**검증 범위**:
- ✅ FileMeta stub
- ✅ validators (mime / 2GB)
- ✅ keying.make_file_checksum 결정성
- ✅ file_manager_client.fetch_file_bytes mock
- ✅ runner.llm_request_async — 파일 없는 케이스 (실제 LLM 호출 + MDB 캐시)
- ✅ runner.llm_request_async — 파일 있는 케이스 (의도된 NotImplementedError)

**검증 불가** (② Phase A 영역):
- classify_files 분기, inline/Files API 실제 처리

테스트 후 이 노트북 삭제 가능.

## 1. 셋업 — sys.path + 환경변수 + file_manager stub

In [1]:
import os
import sys
from pathlib import Path

# service 코드 경로 추가 (quickstarts/ 기준 ../service)
notebook_dir = Path.cwd()
service_root = (notebook_dir.parent / "service").resolve()
sys.path.insert(0, str(service_root))
print(f"service path: {service_root}")

# 실제 인프라 환경변수 설정 (configure_handlers 호출 전)
os.environ["MDB_HANDLER_URL"] = "http://192.7.9.45:8300"
os.environ["LLM_HANDLER_URL"] = "http://192.7.9.45:8100"
os.environ["REDIS_URL"] = "redis://192.7.9.45:7617/0"
os.environ["MONGODB_DATABASE"] = "kw_ai_dev"  # 테스트용 DB (CLAUDE.md 규칙)
os.environ["FILE_MANAGER_URL"] = "http://localhost:8102"  # stub 호출 안 되니 무관
print("env vars set")

service path: /home/rladustn1227/02.github/Kiwoom-Rag/packages/llm-request/service
env vars set


In [2]:
# file_manager 패키지 stub 등록 — service 측 stub의 FileMeta를 차용
import types

# service stub 모듈은 file_manager에 의존하지 않음 (자체 정의)
from app.models.file_meta import FileMeta as _FileMeta

_file_manager_module = types.ModuleType("file_manager")
_file_manager_module.FileMeta = _FileMeta
sys.modules["file_manager"] = _file_manager_module

# 검증
from file_manager import FileMeta
print(f"file_manager.FileMeta = {FileMeta}")
print("stub registered ✓")

file_manager.FileMeta = <class 'app.models.file_meta.FileMeta'>
stub registered ✓


In [3]:
# 외부 핸들러 초기화 (실제 인프라 연결)
from app.core.providers import configure_handlers

configure_handlers()
print("handlers configured ✓ (MongoDB, Redis, llm-handler, file_manager httpx)")

handlers configured ✓ (MongoDB, Redis, llm-handler, file_manager httpx)


## 2. FileMeta 만들기 — small_file.pdf / large_file.pdf

In [4]:
import os
from pathlib import Path

small_path = Path("small_file.pdf").resolve()
large_path = Path("large_file.pdf").resolve()

small_meta = FileMeta(
    file_name="small_file.pdf",
    file_mtime=int(small_path.stat().st_mtime_ns),
    file_size=small_path.stat().st_size,
    file_path=f"/fileserver/{small_path.name}",  # file_server 가정 경로
    mime_type="application/pdf",
    extension="pdf",
)
large_meta = FileMeta(
    file_name="large_file.pdf",
    file_mtime=int(large_path.stat().st_mtime_ns),
    file_size=large_path.stat().st_size,
    file_path=f"/fileserver/{large_path.name}",
    mime_type="application/pdf",
    extension="pdf",
)

print(f"small: {small_meta.file_size:,} bytes")
print(f"large: {large_meta.file_size:,} bytes")
print(small_meta.model_dump())

small: 612,269 bytes
large: 92,716,371 bytes
{'file_name': 'small_file.pdf', 'file_mtime': 1776208755229108999, 'file_size': 612269, 'file_path': '/fileserver/small_file.pdf', 'mime_type': 'application/pdf', 'extension': 'pdf'}


## 3. validators — mime 화이트리스트 + 2GB 초과

In [5]:
from app.logic.file_handler.validators import (
    validate_mime_types,
    validate_file_sizes,
    SUPPORTED_MIME_TYPES,
    MAX_FILE_SIZE,
)
from app.core.errors import UnsupportedFileTypeError, FileTooLargeError

# 정상 케이스
validate_mime_types([small_meta, large_meta])
validate_file_sizes([small_meta, large_meta])
print("정상 케이스 통과 ✓")
print(f"화이트리스트 사이즈: {len(SUPPORTED_MIME_TYPES)}")
print(f"MAX_FILE_SIZE: {MAX_FILE_SIZE / 1024**3:.1f} GB")

정상 케이스 통과 ✓
화이트리스트 사이즈: 26
MAX_FILE_SIZE: 2.0 GB


In [6]:
# 실패 케이스 1 — 지원 안 되는 mime
bad_mime = FileMeta(
    file_name="x.exe",
    file_mtime=1,
    file_size=100,
    file_path="/x",
    mime_type="application/x-msdownload",
    extension="exe",
)
try:
    validate_mime_types([bad_mime])
    print("FAIL: should have raised")
except UnsupportedFileTypeError as e:
    print(f"OK ✓ UnsupportedFileTypeError: {e}")

OK ✓ UnsupportedFileTypeError: 지원하지 않는 파일 타입: mime_type=application/x-msdownload, file_name=x.exe


In [7]:
# 실패 케이스 2 — 2GB 초과
bad_size = FileMeta(
    file_name="huge.pdf",
    file_mtime=1,
    file_size=3 * 1024**3,  # 3GB
    file_path="/h",
    mime_type="application/pdf",
    extension="pdf",
)
try:
    validate_file_sizes([bad_size])
    print("FAIL: should have raised")
except FileTooLargeError as e:
    print(f"OK ✓ FileTooLargeError: {e}")

OK ✓ FileTooLargeError: 파일 크기가 2GB를 초과합니다: file_name=huge.pdf, size=3221225472


## 4. keying.make_file_checksum — 결정성 + 메타 변경 시 변동

In [8]:
from app.logic.keying import make_file_checksum

cs_small_a = make_file_checksum(small_meta)
cs_small_b = make_file_checksum(small_meta)
cs_large = make_file_checksum(large_meta)

print(f"small  checksum #1: {cs_small_a}")
print(f"small  checksum #2: {cs_small_b}")
print(f"large  checksum   : {cs_large}")
assert cs_small_a == cs_small_b, "같은 메타 → 같은 checksum 이어야 함"
assert cs_small_a != cs_large, "다른 메타 → 다른 checksum"
print("결정성 + 차별성 OK ✓")

# 메타 변경 → 다른 checksum
small_modified = small_meta.model_copy(update={"file_mtime": small_meta.file_mtime + 1})
cs_modified = make_file_checksum(small_modified)
assert cs_small_a != cs_modified, "file_mtime 변경 시 checksum 달라야 함"
print(f"mtime 변경 시 다른 checksum ✓")

small  checksum #1: d5c66e29a0ac4eae9518e4b131abe7b316e276980b18e0f717e5a9c3af04c815
small  checksum #2: d5c66e29a0ac4eae9518e4b131abe7b316e276980b18e0f717e5a9c3af04c815
large  checksum   : 0ab13dd5aebfa6e79ed70ab3e028ee41bdbfcc6b1058e39bb8faf2decdd11c35
결정성 + 차별성 OK ✓
mtime 변경 시 다른 checksum ✓


## 5. file_manager_client.fetch_file_bytes — mock으로 로컬 파일 read

In [9]:
import unittest.mock
from app.core import file_manager_client

# fetch_file_bytes mock — file_path의 basename으로 로컬 파일 매칭
async def _fake_fetch(file_path: str) -> bytes:
    name = Path(file_path).name
    local = Path(name).resolve()
    return local.read_bytes()

with unittest.mock.patch.object(file_manager_client, "fetch_file_bytes", _fake_fetch):
    bytes_small = await file_manager_client.fetch_file_bytes(small_meta.file_path)
    bytes_large = await file_manager_client.fetch_file_bytes(large_meta.file_path)

print(f"small bytes: {len(bytes_small):,}")
print(f"large bytes: {len(bytes_large):,}")
assert len(bytes_small) == small_meta.file_size
assert len(bytes_large) == large_meta.file_size
print("fetch_file_bytes mock OK ✓")

small bytes: 612,269
large bytes: 92,716,371
fetch_file_bytes mock OK ✓


## 6. runner.llm_request_async — 파일 없는 요청 (실제 LLM 호출 + MDB 캐시)

실제 Gemini 호출 + MongoDB 캐시 저장/조회 흐름 검증.

In [10]:
from app.logic.runner import llm_request_async

llm_input = {
    "contents": "안녕하세요. '테스트 응답'이라고만 답해주세요.",
    "model": "gemini-2.5-flash",
}

# 첫 호출 — cache miss 예상
result, meta = await llm_request_async(
    llm_input,
    cache_mode=True,
    save_mode=True,
    database="kw_ai_dev",
)
print(f"text     : {result.text}")
print(f"source   : {meta['source']}")
print(f"cache_hit: {meta['cache_hit']}")
print(f"cache_key: {meta['cache_key'][:24]}...")

text     : 테스트 응답
source   : live
cache_hit: False
cache_key: 952fe77c98c5e142249e1e35...


In [11]:
# 두 번째 호출 — 같은 입력이면 cache hit 예상
result2, meta2 = await llm_request_async(
    llm_input,
    cache_mode=True,
    save_mode=True,
    database="kw_ai_dev",
)
print(f"source   : {meta2['source']}")
print(f"cache_hit: {meta2['cache_hit']}")
assert meta2["cache_hit"] is True, "두 번째 호출은 cache hit여야 함"
assert result.text == result2.text
print("cache hit 동작 ✓")

source   : cache
cache_hit: True
cache hit 동작 ✓


## 7. runner.llm_request_async — 파일 있는 요청 (NotImplementedError 의도된 신호)

파일 처리 본문은 ② Phase A에서 구현 예정. 현 단계에선 명시적 신호.

In [12]:
try:
    result, meta = await llm_request_async(
        {
            "contents": "이 파일 한 줄로 요약해줘",
            "model": "gemini-2.5-flash",
        },
        file_metas=[small_meta],
        cache_mode=True,
        save_mode=False,
        database="kw_ai_dev",
    )
    print("FAIL: should have raised NotImplementedError")
except NotImplementedError as e:
    print(f"OK ✓ NotImplementedError (의도된 신호):\n  {e}")

OK ✓ NotImplementedError (의도된 신호):
  phase-② pending: file processing routing not yet implemented (see docs/inline-vs-files-api-strategy.md §4.1, §6)


## 8. 정리

- ① 인터페이스/데이터 흐름 변경이 실제 인프라에서 의도대로 동작
- 파일 없는 요청: 정상 (LLM 호출 + MDB 캐시 hit/miss)
- 파일 있는 요청: 검증·prepare(MDB)까진 정상 → `_build_server_call_input`에서 NotImplementedError
- ② Phase A에서 NotImplementedError 자리를 `classify_files` + `inline/Files API` 분기로 채움

검증 끝나면 이 노트북 삭제 가능.